# 04 · Common Crawl, Step 1: from company to VAT

Research log: [Step 5](../docs/research_log.md#step-5--common-crawl-pasul-1-pornind-de-la-firmă) (in Romanian). Data and checks as of October 2026.

**Question.** If the company website is known, can its VAT number be found in the Common Crawl archive without visiting the site?

**Method.** CC index query per domain (`matchType=domain`), only the needed records fetched with HTTP `Range`, `warcio` + `pypdf`, pages prioritised Terms/Contact/Legal → Privacy → Policy → rest, max. 30 pages per domain, regex "VAT" + 9 digits + checksum. Code versions v1 → v2 → v3 are kept in order.

**Result (from the research log).**
- VAT found: v1 1/12, v2 1/12, v3 2/12 (Laine in a PDF, Rope & Sling on the Contact page); 0 false positives on 172 pages read.
- 10/12 domains are in `CC-MAIN-2026-39`; 5 of 7 sites checked by hand do not publish their VAT number.
- Biased sample: the 12 companies were chosen because Endole had their VAT, so this tests the method, not coverage.

**Notes for re-running.**
- Calls the Common Crawl index (frequent 504s; the code retries).
- v1 writes `cc_pas1_domenii.csv`; the file kept in the repo is `data/results/cc_pas1_domenii_v1.csv`. `cc_pas1_v3_pdf.csv` (written by a v3 cell) is not in the repo.

*Cells are kept as they were run, outputs included; only file paths were updated to the new `data/` layout and session timestamps in outputs were cut to the month.*

In [6]:
import gzip, io, json, re, time, csv
import requests

HEADERS = {'User-Agent': 'vat-research-student (Veridion challenge)'}
CRAWL = None          # None = cel mai recent crawl; sau pune un ID, ex. 'CC-MAIN-2026-xx'
MAX_PAGINI = 30       # cate pagini descarcam maxim per domeniu

# firmele din esantion cu TVA confirmat de HMRC si site cunoscut (Mohari nu are site)
FIRME = [
    # (company_number, firma, domeniu, tva_hmrc)
    ('00307876', 'HAULFRYN GROUP',     'haulfryngroup.com',        '159723633'),
    ('08734445', 'GREENSPRUCE GP',     'invesco.com',              '193725187'),  # site de grup!
    ('10829012', 'WAHED INVEST',       'wahed.com',                '484624270'),
    ('01180133', 'LAINE THEATRE ARTS', 'laine-theatre-arts.co.uk', '211105734'),
    ('09229677', 'BP3 GLOBAL',         'bp-3.com',                 '437464183'),
    ('11488011', 'STRATFORD RETAIL',   'stratfordcapital.co.uk',   '324887179'),
    ('09802945', 'RED CONSTRUCTION',   'redconstruction.co.uk',    '230249147'),
    ('NI030990', 'ENCIRC',             'encirc360.com',            '143776493'),
    ('04084819', 'ROPE & SLING',       'ropeandsling.co.uk',       '811037280'),
    ('09066663', 'ABQ INVESTMENT',     'abqglobal.com',            '208066915'),
    ('11668305', 'JERA NEX',           'jeranex.com',              '315775783'),
    ('09098453', 'ELIZABETH SCHOOL',   'esl.ac.uk',                '264050919'),
]

def tva_valid_checksum(nr):
    nr = nr.replace('GB', '').replace(' ', '')
    if len(nr) != 9 or not nr.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr[:7], range(8, 1, -1))) + int(nr[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

TIPAR = re.compile(r'VAT[^0-9]{0,40}?(?:GB)?\s?(\d{3})\s?(\d{4})\s?(\d{2})(?!\d)', re.I)
PAGINI_PRIORITARE = re.compile(r'contact|terms|privacy|legal|about|imprint|cookie|policy', re.I)

def get(url, **kw):
    """GET cu 3 incercari (serverul Common Crawl da des 503)."""
    for incercare in range(3):
        try:
            r = requests.get(url, headers={**HEADERS, **kw.pop('headers', {})}, timeout=120, **kw)
            if r.status_code in (200, 206, 404):
                return r
        except requests.RequestException:
            pass
        time.sleep(5 * (incercare + 1))
    return None

def pagini_din_index(crawl, domeniu):
    r = get(f'https://index.commoncrawl.org/{crawl}-index',
            params={'url': f'{domeniu}/*', 'output': 'json', 'limit': 1000})
    if r is None or r.status_code == 404:
        return []
    pagini, vazute = [], set()
    for linie in r.text.splitlines():
        p = json.loads(linie)
        if p.get('status') == '200' and 'html' in p.get('mime', '') and p['url'] not in vazute:
            vazute.add(p['url'])
            pagini.append(p)
    # intai paginile cu sanse mari (contact, terms, privacy...), apoi pagina principala, apoi restul
    pagini.sort(key=lambda p: (not PAGINI_PRIORITARE.search(p['url']), len(p['url'])))
    return pagini

def descarca_pagina(p):
    start = int(p['offset']); end = start + int(p['length']) - 1
    r = get(f"https://data.commoncrawl.org/{p['filename']}", headers={'Range': f'bytes={start}-{end}'})
    if r is None:
        return ''
    return gzip.GzipFile(fileobj=io.BytesIO(r.content)).read().decode('utf-8', errors='ignore')

crawl = CRAWL or get('https://index.commoncrawl.org/collinfo.json').json()[0]['id']
print('Crawl:', crawl, '\n')

rezultate = []
for nr_firma, firma, domeniu, tva_hmrc in FIRME:
    t0 = time.time()
    pagini = pagini_din_index(crawl, domeniu)
    gasit, url_gasit, alte, descarcate = False, '', set(), 0
    for p in pagini[:MAX_PAGINI]:
        text = descarca_pagina(p); descarcate += 1
        for m in TIPAR.finditer(text):
            nr = ''.join(m.groups())
            if not tva_valid_checksum(nr):
                continue
            if nr == tva_hmrc:
                gasit, url_gasit = True, p['url']
            else:
                alte.add(nr)
        if gasit:
            break
        time.sleep(1)
    durata = round(time.time() - t0)
    rezultate.append({'company_number': nr_firma, 'firma': firma, 'domeniu': domeniu,
                      'tva_hmrc': tva_hmrc, 'pagini_in_index': len(pagini),
                      'pagini_descarcate': descarcate, 'tva_gasit_cc': gasit,
                      'url_gasit': url_gasit, 'alte_numere': ' '.join(sorted(alte)),
                      'secunde': durata})
    print(f"{firma:20} {domeniu:26} index={len(pagini):4} descarcate={descarcate:3} "
          f"gasit={gasit!s:5} {durata:4}s  {url_gasit}  alte={sorted(alte)}")
    time.sleep(2)

# rezumat + salvare
n = len(rezultate)
prezente = sum(r['pagini_in_index'] > 0 for r in rezultate)
gasite = sum(r['tva_gasit_cc'] for r in rezultate)
print(f'\nDomenii prezente in {crawl}: {prezente}/{n}')
print(f'TVA corect gasit in Common Crawl: {gasite}/{n}')
print(f'Timp total: {sum(r["secunde"] for r in rezultate)//60} min')

with open('../data/results/cc_pas1_domenii.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rezultate[0].keys()))
    w.writeheader(); w.writerows(rezultate)
print('Salvat: data/cc_pas1_domenii.csv')

Crawl: CC-MAIN-2026-39 

HAULFRYN GROUP       haulfryngroup.com          index=   0 descarcate=  0 gasit=False   32s    alte=[]
GREENSPRUCE GP       invesco.com                index= 865 descarcate= 30 gasit=False   59s    alte=[]
WAHED INVEST         wahed.com                  index= 849 descarcate= 30 gasit=False   72s    alte=[]
LAINE THEATRE ARTS   laine-theatre-arts.co.uk   index=  21 descarcate= 21 gasit=False   48s    alte=[]
BP3 GLOBAL           bp-3.com                   index=   0 descarcate=  0 gasit=False   61s    alte=[]
STRATFORD RETAIL     stratfordcapital.co.uk     index=   0 descarcate=  0 gasit=False   12s    alte=[]
RED CONSTRUCTION     redconstruction.co.uk      index=  92 descarcate= 30 gasit=False   56s    alte=[]
ENCIRC               encirc360.com              index=  96 descarcate= 30 gasit=False   53s    alte=[]
ROPE & SLING         ropeandsling.co.uk         index= 168 descarcate=  1 gasit=True     5s  https://www.ropeandsling.co.uk/contact/  alte=[]
ABQ INVES

In [ ]:
import io, json, re, time, csv
from pypdf import PdfReader
import requests
from warcio.archiveiterator import ArchiveIterator

# ------------------------------------------------------------------
# SETARI
# ------------------------------------------------------------------
HEADERS = {'User-Agent': 'vat-research-student (Veridion challenge)'}
CRAWL = 'CC-MAIN-2026-39'   # acelasi crawl ca in v1, ca rezultatele sa fie comparabile
MAX_PAGINI = 30             # cate pagini descarcam maxim per domeniu
DOAR_TEST = True         # True = ruleaza doar pe Wahed si Jera Nex(test rapid)

FIRME = [
    # (company_number, firma, domeniu, tva_hmrc)
    ('00307876', 'HAULFRYN GROUP',     'haulfryngroup.com',        '159723633'),
    ('08734445', 'GREENSPRUCE GP',     'invesco.com',              '193725187'),  # site de grup
    ('10829012', 'WAHED INVEST',       'wahed.com',                '484624270'),
    ('01180133', 'LAINE THEATRE ARTS', 'laine-theatre-arts.co.uk', '211105734'),
    ('09229677', 'BP3 GLOBAL',         'bp-3.com',                 '437464183'),
    ('11488011', 'STRATFORD RETAIL',   'stratfordcapital.co.uk',   '324887179'),
    ('09802945', 'RED CONSTRUCTION',   'redconstruction.co.uk',    '230249147'),
    ('NI030990', 'ENCIRC',             'encirc360.com',            '143776493'),
    ('04084819', 'ROPE & SLING',       'ropeandsling.co.uk',       '811037280'),
    ('09066663', 'ABQ INVESTMENT',     'abqglobal.com',            '208066915'),
    ('11668305', 'JERA NEX',           'jeranex.com',              '315775783'),
    ('09098453', 'ELIZABETH SCHOOL',   'esl.ac.uk',                '264050919'),
]
if DOAR_TEST:
    FIRME = [f for f in FIRME if f[1] in ('WAHED INVEST', 'JERA NEX')]

# ------------------------------------------------------------------
# 1. SUMA DE CONTROL (testata ieri pe cele 13 numere confirmate)
# ------------------------------------------------------------------
def tva_valid_checksum(nr):
    nr = nr.replace('GB', '').replace(' ', '')
    if len(nr) != 9 or not nr.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr[:7], range(8, 1, -1))) + int(nr[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

# 'VAT' urmat (in max 40 caractere) de 9 cifre, cu/fara GB si spatii: GB123456789, 123 4567 89 etc.
TIPAR = re.compile(r'VAT[^0-9]{0,40}?(?:GB)?\s?(\d{3})\s?(\d{4})\s?(\d{2})(?!\d)', re.I)
PAGINI_PRIORITARE = re.compile(r'contact|terms|privacy|legal|about|imprint|cookie|policy', re.I)

def text_curat(html):
    """Scoate tagurile HTML si spatiile multiple, ca sa poti citi contextul."""
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', html))

# ------------------------------------------------------------------
# 2. CERERI HTTP CU REINCERCARI
# ------------------------------------------------------------------
def get(url, params=None, extra_headers=None):
    """GET cu 3 incercari. Intoarce raspunsul sau None daca toate au esuat."""
    headers = {**HEADERS, **(extra_headers or {})}
    for incercare in range(3):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=120)
            if r.status_code in (200, 206, 404):
                return r
            print(f'      status {r.status_code}, reincerc...')
        except requests.RequestException as e:
            print(f'      eroare retea ({type(e).__name__}), reincerc...')
        time.sleep(5 * (incercare + 1))
    return None

# ------------------------------------------------------------------
# 3. INDEXUL: ce pagini are Common Crawl de pe domeniu (inclusiv www. si alte subdomenii)
# ------------------------------------------------------------------
def pagini_din_index(crawl, domeniu):
    r = get(f'https://index.commoncrawl.org/{crawl}-index',
            params={'url': domeniu, 'matchType': 'domain', 'output': 'json', 'limit': 1000})
    if r is None:
        return None                          # EROARE de server (diferit de "nu exista")
    if r.status_code == 404:
        return []                            # domeniul chiar nu e in acest crawl
    pagini, vazute = [], set()
    for linie in r.text.splitlines():
        if not linie.strip():
            continue
        p = json.loads(linie)
        if p.get('status') == '200' and 'html' in p.get('mime', '') and p['url'] not in vazute:
            vazute.add(p['url'])
            pagini.append(p)
    # intai Contact/Terms/Privacy..., apoi URL-urile scurte (pagina principala), apoi restul
    pagini.sort(key=lambda p: (not PAGINI_PRIORITARE.search(p['url']), len(p['url'])))
    return pagini

# ------------------------------------------------------------------
# 4. DESCARCAREA UNEI SINGURE PAGINI DIN ARHIVA
# ------------------------------------------------------------------
def descarca_pagina(p):
    start = int(p['offset']); end = start + int(p['length']) - 1
    r = get(f"https://data.commoncrawl.org/{p['filename']}",
            extra_headers={'Range': f'bytes={start}-{end}'})
    if r is None:
        return None
    for rec in ArchiveIterator(io.BytesIO(r.content)):
        if rec.rec_type == 'response':
            # content_stream() decodeaza si compresia HTTP (gzip / brotli / chunked)
            return rec.content_stream().read().decode('utf-8', errors='ignore')
    return ''

# ------------------------------------------------------------------
# 5. BUCLA PRINCIPALA
# ------------------------------------------------------------------
print('Crawl:', CRAWL, '\n')
rezultate = []

for nr_firma, firma, domeniu, tva_hmrc in FIRME:
    t0 = time.time()
    print(f'--- {firma} ({domeniu})')
    pagini = pagini_din_index(CRAWL, domeniu)

    rand = {'company_number': nr_firma, 'firma': firma, 'domeniu': domeniu, 'tva_hmrc': tva_hmrc,
            'pagini_in_index': '', 'pagini_descarcate': 0, 'pagini_cu_cuvantul_vat': 0,
            'erori_descarcare': 0, 'tva_gasit_cc': False, 'url_gasit': '',
            'alte_numere': '', 'diagnostic': '', 'secunde': 0}

    if pagini is None:
        rand['pagini_in_index'] = 'EROARE'
        rand['diagnostic'] = 'eroare server la index - de reincercat'
    elif not pagini:
        rand['pagini_in_index'] = 0
        rand['diagnostic'] = 'domeniul lipseste din crawl'
    else:
        rand['pagini_in_index'] = len(pagini)
        alte = {}
        for p in pagini[:MAX_PAGINI]:
            html = descarca_pagina(p)
            if html is None:
                rand['erori_descarcare'] += 1
                continue
            rand['pagini_descarcate'] += 1
            if re.search(r'\bVAT\b', html, re.I):
                rand['pagini_cu_cuvantul_vat'] += 1
            for m in TIPAR.finditer(html):
                nr = ''.join(m.groups())
                if not tva_valid_checksum(nr):
                    continue
                context = text_curat(html[max(0, m.start()-200): m.end()+60])[-160:]
                if nr == tva_hmrc:
                    rand['tva_gasit_cc'], rand['url_gasit'] = True, p['url']
                    print(f'    CORECT pe {p["url"]}\n      ...{context}...')
                elif nr not in alte:
                    alte[nr] = p['url']
                    print(f'    ALT NUMAR {nr} pe {p["url"]}\n      ...{context}...')
            if rand['tva_gasit_cc']:
                break
            time.sleep(1)
        rand['alte_numere'] = ' '.join(f'{n}@{u}' for n, u in alte.items())

        # diagnostic: de ce n-am gasit?
        if rand['tva_gasit_cc']:
            rand['diagnostic'] = 'gasit'
        elif rand['pagini_cu_cuvantul_vat'] == 0:
            rand['diagnostic'] = 'nicio pagina descarcata nu contine cuvantul VAT'
        else:
            rand['diagnostic'] = 'apare cuvantul VAT, dar nu numarul firmei - de verificat de mana'

    rand['secunde'] = round(time.time() - t0)
    rezultate.append(rand)
    print(f"    index={rand['pagini_in_index']} descarcate={rand['pagini_descarcate']} "
          f"cu_VAT={rand['pagini_cu_cuvantul_vat']} erori={rand['erori_descarcare']} "
          f"gasit={rand['tva_gasit_cc']} {rand['secunde']}s -> {rand['diagnostic']}\n")
    time.sleep(2)

# ------------------------------------------------------------------
# 6. REZUMAT + SALVARE
# ------------------------------------------------------------------
n = len(rezultate)
erori = sum(r['pagini_in_index'] == 'EROARE' for r in rezultate)
prezente = sum(isinstance(r['pagini_in_index'], int) and r['pagini_in_index'] > 0 for r in rezultate)
gasite = sum(r['tva_gasit_cc'] for r in rezultate)
print(f'Domenii prezente in {CRAWL}: {prezente}/{n}  (erori de server: {erori})')
print(f'TVA corect gasit in Common Crawl: {gasite}/{n}')
print(f'Timp total: {sum(r["secunde"] for r in rezultate) // 60} min')

with open('../data/results/cc_pas1_domenii_v2.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rezultate[0].keys()))
    w.writeheader(); w.writerows(rezultate)
print('Salvat: data/cc_pas1_domenii_v2.csv')

Crawl: CC-MAIN-2026-39 

--- WAHED INVEST (wahed.com)
      status 502, reincerc...
      status 502, reincerc...
      status 504, reincerc...
    index=EROARE descarcate=0 cu_VAT=0 erori=0 gasit=False 59s -> eroare server la index - de reincercat

--- JERA NEX (jeranex.com)
      status 502, reincerc...
    index=17 descarcate=17 cu_VAT=0 erori=0 gasit=False 36s -> nicio pagina descarcata nu contine cuvantul VAT

Domenii prezente in CC-MAIN-2026-39: 1/2  (erori de server: 1)
TVA corect gasit in Common Crawl: 0/2
Timp total: 1 min
Salvat: data/cc_pas1_domenii_v2.csv


In [1]:
import io, json, re, time, csv
import requests
from pypdf import PdfReader
from warcio.archiveiterator import ArchiveIterator

# ------------------------------------------------------------------
# SETARI
# ------------------------------------------------------------------
HEADERS = {'User-Agent': 'vat-research-student (Veridion challenge)'}
CRAWL = 'CC-MAIN-2026-39'
MAX_PAGINI = 30
FIRME_TEST = ['LAINE THEATRE ARTS']      # lista goala [] = ruleaza pe toate 12
FISIER_IESIRE = '../data/results/cc_pas1_v3_pdf.csv'

FIRME = [
    # (company_number, firma, domeniu, tva_hmrc)
    ('00307876', 'HAULFRYN GROUP',     'haulfryngroup.com',        '159723633'),
    ('08734445', 'GREENSPRUCE GP',     'invesco.com',              '193725187'),  # site de grup
    ('10829012', 'WAHED INVEST',       'wahed.com',                '484624270'),
    ('01180133', 'LAINE THEATRE ARTS', 'laine-theatre-arts.co.uk', '211105734'),
    ('09229677', 'BP3 GLOBAL',         'bp-3.com',                 '437464183'),
    ('11488011', 'STRATFORD RETAIL',   'stratfordcapital.co.uk',   '324887179'),
    ('09802945', 'RED CONSTRUCTION',   'redconstruction.co.uk',    '230249147'),
    ('NI030990', 'ENCIRC',             'encirc360.com',            '143776493'),
    ('04084819', 'ROPE & SLING',       'ropeandsling.co.uk',       '811037280'),
    ('09066663', 'ABQ INVESTMENT',     'abqglobal.com',            '208066915'),
    ('11668305', 'JERA NEX',           'jeranex.com',              '315775783'),
    ('09098453', 'ELIZABETH SCHOOL',   'esl.ac.uk',                '264050919'),
]
if FIRME_TEST:
    FIRME = [f for f in FIRME if f[1] in FIRME_TEST]

# ------------------------------------------------------------------
# 1. SUMA DE CONTROL SI TIPARE
# ------------------------------------------------------------------
def tva_valid_checksum(nr):
    nr = nr.replace('GB', '').replace(' ', '')
    if len(nr) != 9 or not nr.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr[:7], range(8, 1, -1))) + int(nr[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

TIPAR = re.compile(r'VAT[^0-9]{0,40}?(?:GB)?\s?(\d{3})\s?(\d{4})\s?(\d{2})(?!\d)', re.I)
PAGINI_PRIORITARE = re.compile(r'contact|terms|privacy|legal|about|imprint|cookie|policy|polic|condition', re.I)

def text_curat(text):
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', text))

def numar_firma_in_context(company_number, context):
    """True daca numarul Companies House apare langa TVA (cu sau fara zerourile din fata)."""
    fara_zero = company_number.lstrip('0')
    return bool(re.search(rf'(?<!\d)0*{re.escape(fara_zero)}(?!\d)', context))

# ------------------------------------------------------------------
# 2. CERERI HTTP CU REINCERCARI
# ------------------------------------------------------------------
def get(url, params=None, extra_headers=None):
    headers = {**HEADERS, **(extra_headers or {})}
    for incercare in range(3):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=120)
            if r.status_code in (200, 206, 404):
                return r
            print(f'      status {r.status_code}, reincerc...')
        except requests.RequestException as e:
            print(f'      eroare retea ({type(e).__name__}), reincerc...')
        time.sleep(5 * (incercare + 1))
    return None

# ------------------------------------------------------------------
# 3. INDEXUL: pagini HTML si PDF de pe domeniu (inclusiv subdomenii)
# ------------------------------------------------------------------
def pagini_din_index(crawl, domeniu):
    r = get(f'https://index.commoncrawl.org/{crawl}-index',
            params={'url': domeniu, 'matchType': 'domain', 'output': 'json', 'limit': 1000})
    if r is None:
        return None
    if r.status_code == 404:
        return []
    pagini, vazute = [], set()
    for linie in r.text.splitlines():
        if not linie.strip():
            continue
        p = json.loads(linie)
        mime = p.get('mime', '')
        if p.get('status') == '200' and ('html' in mime or 'pdf' in mime) and p['url'] not in vazute:
            vazute.add(p['url'])
            pagini.append(p)
    # ordinea: 1) Contact/Terms/Policy..., 2) PDF-uri, 3) URL-uri scurte (pagina principala), 4) restul
    pagini.sort(key=lambda p: (not PAGINI_PRIORITARE.search(p['url']),
                               'pdf' not in p.get('mime', ''),
                               len(p['url'])))
    return pagini

# ------------------------------------------------------------------
# 4. DESCARCAREA UNEI PAGINI (HTML sau PDF) DIN ARHIVA
# ------------------------------------------------------------------
def descarca_pagina(p):
    start = int(p['offset']); end = start + int(p['length']) - 1
    r = get(f"https://data.commoncrawl.org/{p['filename']}",
            extra_headers={'Range': f'bytes={start}-{end}'})
    if r is None:
        return None
    for rec in ArchiveIterator(io.BytesIO(r.content)):
        if rec.rec_type != 'response':
            continue
        continut = rec.content_stream().read()
        if 'pdf' in p.get('mime', ''):
            try:
                pdf = PdfReader(io.BytesIO(continut))
                return ' '.join((pag.extract_text() or '') for pag in pdf.pages)
            except Exception:
                return 'PDF_ILIZIBIL'
        return continut.decode('utf-8', errors='ignore')
    return ''

# ------------------------------------------------------------------
# 5. BUCLA PRINCIPALA
# ------------------------------------------------------------------
print('Crawl:', CRAWL, '\n')
rezultate = []

for nr_firma, firma, domeniu, tva_hmrc in FIRME:
    t0 = time.time()
    print(f'--- {firma} ({domeniu})')
    pagini = pagini_din_index(CRAWL, domeniu)

    rand = {'company_number': nr_firma, 'firma': firma, 'domeniu': domeniu, 'tva_hmrc': tva_hmrc,
            'pagini_in_index': '', 'pdf_in_index': 0, 'pagini_descarcate': 0,
            'pagini_cu_cuvantul_vat': 0, 'pdf_ilizibile': 0, 'erori_descarcare': 0,
            'tva_gasit_cc': False, 'url_gasit': '', 'tip_pagina': '',
            'nr_firma_langa_tva': False, 'alte_numere': '', 'diagnostic': '', 'secunde': 0}

    if pagini is None:
        rand['pagini_in_index'] = 'EROARE'
        rand['diagnostic'] = 'eroare server la index - de reincercat'
    elif not pagini:
        rand['pagini_in_index'] = 0
        rand['diagnostic'] = 'domeniul lipseste din crawl'
    else:
        rand['pagini_in_index'] = len(pagini)
        rand['pdf_in_index'] = sum('pdf' in p.get('mime', '') for p in pagini)
        alte = {}
        for p in pagini[:MAX_PAGINI]:
            text = descarca_pagina(p)
            if text is None:
                rand['erori_descarcare'] += 1
                continue
            if text == 'PDF_ILIZIBIL':
                rand['pdf_ilizibile'] += 1
                print(f'    PDF ilizibil (posibil trunchiat): {p["url"]}')
                continue

Crawl: CC-MAIN-2026-39 

--- LAINE THEATRE ARTS (laine-theatre-arts.co.uk)
      status 504, reincerc...


Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 105 0 (offset 0)


In [2]:
PDF_TINTA = 'laine-theatre-arts.co.uk/wp-content/uploads/2026/06/2026-DIPLOMA-DEGREE-COURSE-TERMS-AND-CONDITIONS-HH-SC.pdf'

# 1. e PDF-ul in crawl-ul nostru? Il cautam si in ultimele crawl-uri
crawls = [c['id'] for c in get('https://index.commoncrawl.org/collinfo.json').json()[:4]]
for c in crawls:
    r = get(f'https://index.commoncrawl.org/{c}-index',
            params={'url': PDF_TINTA, 'output': 'json'})
    if r is None:
        print(c, '-> EROARE server')
    elif r.status_code == 404:
        print(c, '-> PDF-ul NU e in acest crawl')
    else:
        inreg = [json.loads(l) for l in r.text.splitlines() if l.strip()]
        print(c, '-> GASIT:', [(i['status'], i.get('mime'), i['length']) for i in inreg])
    time.sleep(2)

# 2. ce PDF-uri are crawl-ul de pe site si care au fost citite (primele 30)
pagini = pagini_din_index(CRAWL, 'laine-theatre-arts.co.uk')
if pagini:
    pdfuri = [p for p in pagini if 'pdf' in p.get('mime', '')]
    print(f'\nIn {CRAWL}: {len(pagini)} pagini, din care {len(pdfuri)} PDF-uri')
    for i, p in enumerate(pagini[:MAX_PAGINI]):
        print(f"{i:2} {'PDF ' if 'pdf' in p.get('mime','') else 'HTML'} {p['url']}")

      status 504, reincerc...
CC-MAIN-2026-39 -> GASIT: [('200', 'application/pdf', '194173')]
CC-MAIN-2026-34 -> PDF-ul NU e in acest crawl
CC-MAIN-2026-30 -> PDF-ul NU e in acest crawl
CC-MAIN-2026-25 -> PDF-ul NU e in acest crawl

In CC-MAIN-2026-39: 94 pagini, din care 73 PDF-uri
 0 PDF  https://www.laine-theatre-arts.co.uk/policies/UoP-5-Student-Voice-Policy.pdf
 1 PDF  https://www.laine-theatre-arts.co.uk/policies/UoP-3-External-Examiner-Policy.pdf
 2 PDF  https://www.laine-theatre-arts.co.uk/wp-content/uploads/2026/08/A39-Casting-Policy-v1.1.pdf
 3 PDF  https://www.laine-theatre-arts.co.uk/policies/UoP-4-Recognition-of-Higher-Learning-Policy.pdf
 4 PDF  https://www.laine-theatre-arts.co.uk/wp-content/uploads/2026/02/A16-LTA-SEND-Policy-NEW-v1.pdf
 5 PDF  https://www.laine-theatre-arts.co.uk/wp-content/uploads/2026/01/S14LTASexualMisconductPolicyNEW.pdf
 6 PDF  https://www.laine-theatre-arts.co.uk/wp-content/uploads/2026/02/A1-LTA-Admissions-Policy-NEW-v1.pdf
 7 PDF  https://www.

In [3]:
import io, json, re, time, csv, logging
import requests
from pypdf import PdfReader
from warcio.archiveiterator import ArchiveIterator

logging.getLogger('pypdf').setLevel(logging.ERROR)   # ascunde avertismentele PDF

# ------------------------------------------------------------------
# SETARI
# ------------------------------------------------------------------
HEADERS = {'User-Agent': 'vat-research-student (Veridion challenge)'}
CRAWL = 'CC-MAIN-2026-39'
MAX_PAGINI = 30
FIRME_TEST = ['LAINE THEATRE ARTS']      # lista goala [] = ruleaza pe toate 12
FISIER_IESIRE = '../data/results/cc_pas1_v3_laine.csv'

FIRME = [
    # (company_number, firma, domeniu, tva_hmrc)
    ('00307876', 'HAULFRYN GROUP',     'haulfryngroup.com',        '159723633'),
    ('08734445', 'GREENSPRUCE GP',     'invesco.com',              '193725187'),  # site de grup
    ('10829012', 'WAHED INVEST',       'wahed.com',                '484624270'),
    ('01180133', 'LAINE THEATRE ARTS', 'laine-theatre-arts.co.uk', '211105734'),
    ('09229677', 'BP3 GLOBAL',         'bp-3.com',                 '437464183'),
    ('11488011', 'STRATFORD RETAIL',   'stratfordcapital.co.uk',   '324887179'),
    ('09802945', 'RED CONSTRUCTION',   'redconstruction.co.uk',    '230249147'),
    ('NI030990', 'ENCIRC',             'encirc360.com',            '143776493'),
    ('04084819', 'ROPE & SLING',       'ropeandsling.co.uk',       '811037280'),
    ('09066663', 'ABQ INVESTMENT',     'abqglobal.com',            '208066915'),
    ('11668305', 'JERA NEX',           'jeranex.com',              '315775783'),
    ('09098453', 'ELIZABETH SCHOOL',   'esl.ac.uk',                '264050919'),
]
if FIRME_TEST:
    FIRME = [f for f in FIRME if f[1] in FIRME_TEST]

# ------------------------------------------------------------------
# 1. SUMA DE CONTROL, TIPARE, PRIORITATE
# ------------------------------------------------------------------
def tva_valid_checksum(nr):
    nr = nr.replace('GB', '').replace(' ', '')
    if len(nr) != 9 or not nr.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr[:7], range(8, 1, -1))) + int(nr[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

TIPAR = re.compile(r'VAT[^0-9]{0,40}?(?:GB)?\s?(\d{3})\s?(\d{4})\s?(\d{2})(?!\d)', re.I)

def scor_prioritate(url):
    u = url.lower()
    if re.search(r'terms|condition|contact|legal|imprint|about-us|company-info', u):
        return 0      # cele mai probabile locuri pentru TVA
    if re.search(r'privacy|cookie|disclaimer', u):
        return 1
    if re.search(r'polic', u):
        return 2      # politici generale - mai rar
    return 3

def text_curat(text):
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', text))

def numar_firma_in_context(company_number, context):
    """True daca numarul Companies House apare langa TVA (cu sau fara zerourile din fata)."""
    fara_zero = company_number.lstrip('0')
    return bool(re.search(rf'(?<!\d)0*{re.escape(fara_zero)}(?!\d)', context))

# ------------------------------------------------------------------
# 2. CERERI HTTP CU REINCERCARI
# ------------------------------------------------------------------
def get(url, params=None, extra_headers=None):
    headers = {**HEADERS, **(extra_headers or {})}
    for incercare in range(3):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=120)
            if r.status_code in (200, 206, 404):
                return r
            print(f'      status {r.status_code}, reincerc...')
        except requests.RequestException as e:
            print(f'      eroare retea ({type(e).__name__}), reincerc...')
        time.sleep(5 * (incercare + 1))
    return None

# ------------------------------------------------------------------
# 3. INDEXUL: pagini HTML si PDF de pe domeniu (inclusiv subdomenii)
# ------------------------------------------------------------------
def pagini_din_index(crawl, domeniu):
    r = get(f'https://index.commoncrawl.org/{crawl}-index',
            params={'url': domeniu, 'matchType': 'domain', 'output': 'json', 'limit': 1000})
    if r is None:
        return None
    if r.status_code == 404:
        return []
    pagini, vazute = [], set()
    for linie in r.text.splitlines():
        if not linie.strip():
            continue
        p = json.loads(linie)
        mime = p.get('mime', '')
        if p.get('status') == '200' and ('html' in mime or 'pdf' in mime) and p['url'] not in vazute:
            vazute.add(p['url'])
            pagini.append(p)
    pagini.sort(key=lambda p: (scor_prioritate(p['url']), 'pdf' not in p.get('mime', ''), len(p['url'])))
    return pagini

# ------------------------------------------------------------------
# 4. DESCARCAREA UNEI PAGINI (HTML sau PDF) DIN ARHIVA
# ------------------------------------------------------------------
def descarca_pagina(p):
    start = int(p['offset']); end = start + int(p['length']) - 1
    r = get(f"https://data.commoncrawl.org/{p['filename']}",
            extra_headers={'Range': f'bytes={start}-{end}'})
    if r is None:
        return None
    for rec in ArchiveIterator(io.BytesIO(r.content)):
        if rec.rec_type != 'response':
            continue
        continut = rec.content_stream().read()
        if 'pdf' in p.get('mime', ''):
            try:
                pdf = PdfReader(io.BytesIO(continut))
                return ' '.join((pag.extract_text() or '') for pag in pdf.pages)
            except Exception:
                return 'PDF_ILIZIBIL'
        return continut.decode('utf-8', errors='ignore')
    return ''

# ------------------------------------------------------------------
# 5. BUCLA PRINCIPALA
# ------------------------------------------------------------------
print('Crawl:', CRAWL, '\n')
rezultate = []

for nr_firma, firma, domeniu, tva_hmrc in FIRME:
    t0 = time.time()
    print(f'--- {firma} ({domeniu})')
    pagini = pagini_din_index(CRAWL, domeniu)

    rand = {'company_number': nr_firma, 'firma': firma, 'domeniu': domeniu, 'tva_hmrc': tva_hmrc,
            'pagini_in_index': '', 'pdf_in_index': 0, 'pagini_descarcate': 0,
            'pagini_cu_cuvantul_vat': 0, 'pdf_ilizibile': 0, 'erori_descarcare': 0,
            'tva_gasit_cc': False, 'url_gasit': '', 'tip_pagina': '', 'pozitie_in_lista': '',
            'nr_firma_langa_tva': False, 'alte_numere': '', 'diagnostic': '', 'secunde': 0}

    if pagini is None:
        rand['pagini_in_index'] = 'EROARE'
        rand['diagnostic'] = 'eroare server la index - de reincercat'
    elif not pagini:
        rand['pagini_in_index'] = 0
        rand['diagnostic'] = 'domeniul lipseste din crawl'
    else:
        rand['pagini_in_index'] = len(pagini)
        rand['pdf_in_index'] = sum('pdf' in p.get('mime', '') for p in pagini)
        alte = {}
        for i, p in enumerate(pagini[:MAX_PAGINI]):
            text = descarca_pagina(p)
            if text is None:
                rand['erori_descarcare'] += 1
                continue
            if text == 'PDF_ILIZIBIL':
                rand['pdf_ilizibile'] += 1
                print(f'    PDF ilizibil (posibil trunchiat): {p["url"]}')
                continue
            rand['pagini_descarcate'] += 1
            if re.search(r'\bVAT\b', text, re.I):
                rand['pagini_cu_cuvantul_vat'] += 1
            for m in TIPAR.finditer(text):
                nr = ''.join(m.groups())
                if not tva_valid_checksum(nr):
                    continue
                context = text_curat(text[max(0, m.start()-250): m.end()+80])[-220:]
                if nr == tva_hmrc:
                    rand['tva_gasit_cc'], rand['url_gasit'] = True, p['url']
                    rand['tip_pagina'] = 'pdf' if 'pdf' in p.get('mime', '') else 'html'
                    rand['pozitie_in_lista'] = i
                    rand['nr_firma_langa_tva'] = numar_firma_in_context(nr_firma, context)
                    print(f'    CORECT ({rand["tip_pagina"]}, pagina nr. {i}) pe {p["url"]}\n      ...{context}...')
                elif nr not in alte:
                    alte[nr] = p['url']
                    print(f'    ALT NUMAR {nr} pe {p["url"]}\n      ...{context}...')
            if rand['tva_gasit_cc']:
                break
            time.sleep(1)
        rand['alte_numere'] = ' '.join(f'{n}@{u}' for n, u in alte.items())

        if rand['tva_gasit_cc']:
            rand['diagnostic'] = 'gasit'
        elif rand['pagini_cu_cuvantul_vat'] == 0:
            rand['diagnostic'] = 'nicio pagina descarcata nu contine cuvantul VAT'
        else:
            rand['diagnostic'] = 'apare cuvantul VAT, dar nu numarul firmei - de verificat de mana'

    rand['secunde'] = round(time.time() - t0)
    rezultate.append(rand)
    print(f"    index={rand['pagini_in_index']} (pdf={rand['pdf_in_index']}) "
          f"descarcate={rand['pagini_descarcate']} cu_VAT={rand['pagini_cu_cuvantul_vat']} "
          f"pdf_ilizibile={rand['pdf_ilizibile']} erori={rand['erori_descarcare']} "
          f"gasit={rand['tva_gasit_cc']} nr_firma_langa={rand['nr_firma_langa_tva']} "
          f"{rand['secunde']}s -> {rand['diagnostic']}\n")
    time.sleep(2)

# ------------------------------------------------------------------
# 6. REZUMAT + SALVARE
# ------------------------------------------------------------------
n = len(rezultate)
erori = sum(r['pagini_in_index'] == 'EROARE' for r in rezultate)
prezente = sum(isinstance(r['pagini_in_index'], int) and r['pagini_in_index'] > 0 for r in rezultate)
gasite = sum(r['tva_gasit_cc'] for r in rezultate)
print(f'Domenii prezente in {CRAWL}: {prezente}/{n}  (erori de server: {erori})')
print(f'TVA corect gasit in Common Crawl: {gasite}/{n}')
print(f'Timp total: {sum(r["secunde"] for r in rezultate) // 60} min')

with open(FISIER_IESIRE, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rezultate[0].keys()))
    w.writeheader(); w.writerows(rezultate)
print('Salvat:', FISIER_IESIRE)

Crawl: CC-MAIN-2026-39 

--- LAINE THEATRE ARTS (laine-theatre-arts.co.uk)
    CORECT (pdf, pagina nr. 0) pe https://www.laine-theatre-arts.co.uk/wp-content/uploads/2026/06/2026-DIPLOMA-DEGREE-COURSE-TERMS-AND-CONDITIONS-HH-SC.pdf
      ...s Limited is a private company limited by shares, incorporated in England and Wales with registered number 1180133 and VAT Reg. No. 211105734. Our registered office is located at 1st Floor, Sheraton House, Lower Road, Ch...
    index=94 (pdf=73) descarcate=1 cu_VAT=1 pdf_ilizibile=0 erori=0 gasit=True nr_firma_langa=True 8s -> gasit

Domenii prezente in CC-MAIN-2026-39: 1/1  (erori de server: 0)
TVA corect gasit in Common Crawl: 1/1
Timp total: 0 min
Salvat: ../data/cc_pas1_v3_laine.csv


In [4]:
import io, json, re, time, csv, logging
import requests
from pypdf import PdfReader
from warcio.archiveiterator import ArchiveIterator

logging.getLogger('pypdf').setLevel(logging.ERROR)   # ascunde avertismentele PDF

# ------------------------------------------------------------------
# SETARI
# ------------------------------------------------------------------
HEADERS = {'User-Agent': 'vat-research-student (Veridion challenge)'}
CRAWL = 'CC-MAIN-2026-39'
MAX_PAGINI = 30
FIRME_TEST = []                                      # <- lista goala = toate 12 firmele
FISIER_IESIRE = '../data/results/cc_pas1_v3_toate.csv'       # <- fisier nou, nu suprascrie Laine

FIRME = [
    # (company_number, firma, domeniu, tva_hmrc)
    ('00307876', 'HAULFRYN GROUP',     'haulfryngroup.com',        '159723633'),
    ('08734445', 'GREENSPRUCE GP',     'invesco.com',              '193725187'),  # site de grup
    ('10829012', 'WAHED INVEST',       'wahed.com',                '484624270'),
    ('01180133', 'LAINE THEATRE ARTS', 'laine-theatre-arts.co.uk', '211105734'),
    ('09229677', 'BP3 GLOBAL',         'bp-3.com',                 '437464183'),
    ('11488011', 'STRATFORD RETAIL',   'stratfordcapital.co.uk',   '324887179'),
    ('09802945', 'RED CONSTRUCTION',   'redconstruction.co.uk',    '230249147'),
    ('NI030990', 'ENCIRC',             'encirc360.com',            '143776493'),
    ('04084819', 'ROPE & SLING',       'ropeandsling.co.uk',       '811037280'),
    ('09066663', 'ABQ INVESTMENT',     'abqglobal.com',            '208066915'),
    ('11668305', 'JERA NEX',           'jeranex.com',              '315775783'),
    ('09098453', 'ELIZABETH SCHOOL',   'esl.ac.uk',                '264050919'),
]
if FIRME_TEST:
    FIRME = [f for f in FIRME if f[1] in FIRME_TEST]

# ------------------------------------------------------------------
# 1. SUMA DE CONTROL, TIPARE, PRIORITATE
# ------------------------------------------------------------------
def tva_valid_checksum(nr):
    nr = nr.replace('GB', '').replace(' ', '')
    if len(nr) != 9 or not nr.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr[:7], range(8, 1, -1))) + int(nr[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

TIPAR = re.compile(r'VAT[^0-9]{0,40}?(?:GB)?\s?(\d{3})\s?(\d{4})\s?(\d{2})(?!\d)', re.I)

def scor_prioritate(url):
    u = url.lower()
    if re.search(r'terms|condition|contact|legal|imprint|about-us|company-info', u):
        return 0
    if re.search(r'privacy|cookie|disclaimer', u):
        return 1
    if re.search(r'polic', u):
        return 2
    return 3

def text_curat(text):
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', text))

def numar_firma_in_context(company_number, context):
    fara_zero = company_number.lstrip('0')
    return bool(re.search(rf'(?<!\d)0*{re.escape(fara_zero)}(?!\d)', context))

# ------------------------------------------------------------------
# 2. CERERI HTTP CU REINCERCARI
# ------------------------------------------------------------------
def get(url, params=None, extra_headers=None):
    headers = {**HEADERS, **(extra_headers or {})}
    for incercare in range(3):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=120)
            if r.status_code in (200, 206, 404):
                return r
            print(f'      status {r.status_code}, reincerc...')
        except requests.RequestException as e:
            print(f'      eroare retea ({type(e).__name__}), reincerc...')
        time.sleep(5 * (incercare + 1))
    return None

# ------------------------------------------------------------------
# 3. INDEXUL: pagini HTML si PDF de pe domeniu (inclusiv subdomenii)
# ------------------------------------------------------------------
def pagini_din_index(crawl, domeniu):
    r = get(f'https://index.commoncrawl.org/{crawl}-index',
            params={'url': domeniu, 'matchType': 'domain', 'output': 'json', 'limit': 1000})
    if r is None:
        return None
    if r.status_code == 404:
        return []
    pagini, vazute = [], set()
    for linie in r.text.splitlines():
        if not linie.strip():
            continue
        p = json.loads(linie)
        mime = p.get('mime', '')
        if p.get('status') == '200' and ('html' in mime or 'pdf' in mime) and p['url'] not in vazute:
            vazute.add(p['url'])
            pagini.append(p)
    pagini.sort(key=lambda p: (scor_prioritate(p['url']), 'pdf' not in p.get('mime', ''), len(p['url'])))
    return pagini

# ------------------------------------------------------------------
# 4. DESCARCAREA UNEI PAGINI (HTML sau PDF) DIN ARHIVA
# ------------------------------------------------------------------
def descarca_pagina(p):
    start = int(p['offset']); end = start + int(p['length']) - 1
    r = get(f"https://data.commoncrawl.org/{p['filename']}",
            extra_headers={'Range': f'bytes={start}-{end}'})
    if r is None:
        return None
    for rec in ArchiveIterator(io.BytesIO(r.content)):
        if rec.rec_type != 'response':
            continue
        continut = rec.content_stream().read()
        if 'pdf' in p.get('mime', ''):
            try:
                pdf = PdfReader(io.BytesIO(continut))
                return ' '.join((pag.extract_text() or '') for pag in pdf.pages)
            except Exception:
                return 'PDF_ILIZIBIL'
        return continut.decode('utf-8', errors='ignore')
    return ''

# ------------------------------------------------------------------
# 5. BUCLA PRINCIPALA
# ------------------------------------------------------------------
print('Crawl:', CRAWL, '\n')
rezultate = []

for nr_firma, firma, domeniu, tva_hmrc in FIRME:
    t0 = time.time()
    print(f'--- {firma} ({domeniu})')
    pagini = pagini_din_index(CRAWL, domeniu)

    rand = {'company_number': nr_firma, 'firma': firma, 'domeniu': domeniu, 'tva_hmrc': tva_hmrc,
            'pagini_in_index': '', 'pdf_in_index': 0, 'pagini_descarcate': 0,
            'pagini_cu_cuvantul_vat': 0, 'pdf_ilizibile': 0, 'erori_descarcare': 0,
            'tva_gasit_cc': False, 'url_gasit': '', 'tip_pagina': '', 'pozitie_in_lista': '',
            'nr_firma_langa_tva': False, 'alte_numere': '', 'diagnostic': '', 'secunde': 0}

    if pagini is None:
        rand['pagini_in_index'] = 'EROARE'
        rand['diagnostic'] = 'eroare server la index - de reincercat'
    elif not pagini:
        rand['pagini_in_index'] = 0
        rand['diagnostic'] = 'domeniul lipseste din crawl'
    else:
        rand['pagini_in_index'] = len(pagini)
        rand['pdf_in_index'] = sum('pdf' in p.get('mime', '') for p in pagini)
        alte = {}
        for i, p in enumerate(pagini[:MAX_PAGINI]):
            text = descarca_pagina(p)
            if text is None:
                rand['erori_descarcare'] += 1
                continue
            if text == 'PDF_ILIZIBIL':
                rand['pdf_ilizibile'] += 1
                print(f'    PDF ilizibil (posibil trunchiat): {p["url"]}')
                continue
            rand['pagini_descarcate'] += 1
            if re.search(r'\bVAT\b', text, re.I):
                rand['pagini_cu_cuvantul_vat'] += 1
            for m in TIPAR.finditer(text):
                nr = ''.join(m.groups())
                if not tva_valid_checksum(nr):
                    continue
                context = text_curat(text[max(0, m.start()-250): m.end()+80])[-220:]
                if nr == tva_hmrc:
                    rand['tva_gasit_cc'], rand['url_gasit'] = True, p['url']
                    rand['tip_pagina'] = 'pdf' if 'pdf' in p.get('mime', '') else 'html'
                    rand['pozitie_in_lista'] = i
                    rand['nr_firma_langa_tva'] = numar_firma_in_context(nr_firma, context)
                    print(f'    CORECT ({rand["tip_pagina"]}, pagina nr. {i}) pe {p["url"]}\n      ...{context}...')
                elif nr not in alte:
                    alte[nr] = p['url']
                    print(f'    ALT NUMAR {nr} pe {p["url"]}\n      ...{context}...')
            if rand['tva_gasit_cc']:
                break
            time.sleep(1)
        rand['alte_numere'] = ' '.join(f'{n}@{u}' for n, u in alte.items())

        if rand['tva_gasit_cc']:
            rand['diagnostic'] = 'gasit'
        elif rand['pagini_cu_cuvantul_vat'] == 0:
            rand['diagnostic'] = 'nicio pagina descarcata nu contine cuvantul VAT'
        else:
            rand['diagnostic'] = 'apare cuvantul VAT, dar nu numarul firmei - de verificat de mana'

    rand['secunde'] = round(time.time() - t0)
    rezultate.append(rand)
    print(f"    index={rand['pagini_in_index']} (pdf={rand['pdf_in_index']}) "
          f"descarcate={rand['pagini_descarcate']} cu_VAT={rand['pagini_cu_cuvantul_vat']} "
          f"pdf_ilizibile={rand['pdf_ilizibile']} erori={rand['erori_descarcare']} "
          f"gasit={rand['tva_gasit_cc']} nr_firma_langa={rand['nr_firma_langa_tva']} "
          f"{rand['secunde']}s -> {rand['diagnostic']}\n")
    time.sleep(2)

# ------------------------------------------------------------------
# 6. REZUMAT + SALVARE
# ------------------------------------------------------------------
n = len(rezultate)
erori = sum(r['pagini_in_index'] == 'EROARE' for r in rezultate)
prezente = sum(isinstance(r['pagini_in_index'], int) and r['pagini_in_index'] > 0 for r in rezultate)
gasite = sum(r['tva_gasit_cc'] for r in rezultate)
print(f'Domenii prezente in {CRAWL}: {prezente}/{n}  (erori de server: {erori})')
print(f'TVA corect gasit in Common Crawl: {gasite}/{n}')
print(f'Timp total: {sum(r["secunde"] for r in rezultate) // 60} min')

with open(FISIER_IESIRE, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rezultate[0].keys()))
    w.writeheader(); w.writerows(rezultate)
print('Salvat:', FISIER_IESIRE)

Crawl: CC-MAIN-2026-39 

--- HAULFRYN GROUP (haulfryngroup.com)
      status 504, reincerc...
    index=0 (pdf=0) descarcate=0 cu_VAT=0 pdf_ilizibile=0 erori=0 gasit=False nr_firma_langa=False 24s -> domeniul lipseste din crawl

--- GREENSPRUCE GP (invesco.com)
      status 504, reincerc...
      status 504, reincerc...
      eroare retea (ChunkedEncodingError), reincerc...
    index=EROARE (pdf=0) descarcate=0 cu_VAT=0 pdf_ilizibile=0 erori=0 gasit=False nr_firma_langa=False 74s -> eroare server la index - de reincercat

--- WAHED INVEST (wahed.com)
    index=858 (pdf=0) descarcate=30 cu_VAT=1 pdf_ilizibile=0 erori=0 gasit=False nr_firma_langa=False 71s -> apare cuvantul VAT, dar nu numarul firmei - de verificat de mana

--- LAINE THEATRE ARTS (laine-theatre-arts.co.uk)
    CORECT (pdf, pagina nr. 0) pe https://www.laine-theatre-arts.co.uk/wp-content/uploads/2026/06/2026-DIPLOMA-DEGREE-COURSE-TERMS-AND-CONDITIONS-HH-SC.pdf
      ...s Limited is a private company limited by shares, inc

In [6]:
import io, json, re, time, csv, logging
import requests
from pypdf import PdfReader
from warcio.archiveiterator import ArchiveIterator

logging.getLogger('pypdf').setLevel(logging.ERROR)   # ascunde avertismentele PDF

# ------------------------------------------------------------------
# SETARI
# ------------------------------------------------------------------
HEADERS = {'User-Agent': 'vat-research-student (Veridion challenge)'}
CRAWL = 'CC-MAIN-2026-39'
MAX_PAGINI = 30
FIRME_TEST = ['GREENSPRUCE GP', 'BP3 GLOBAL']
FISIER_IESIRE = '../data/results/cc_pas1_v3_retry.csv'

FIRME = [
    # (company_number, firma, domeniu, tva_hmrc)
    ('00307876', 'HAULFRYN GROUP',     'haulfryngroup.com',        '159723633'),
    ('08734445', 'GREENSPRUCE GP',     'invesco.com',              '193725187'),  # site de grup
    ('10829012', 'WAHED INVEST',       'wahed.com',                '484624270'),
    ('01180133', 'LAINE THEATRE ARTS', 'laine-theatre-arts.co.uk', '211105734'),
    ('09229677', 'BP3 GLOBAL',         'bp-3.com',                 '437464183'),
    ('11488011', 'STRATFORD RETAIL',   'stratfordcapital.co.uk',   '324887179'),
    ('09802945', 'RED CONSTRUCTION',   'redconstruction.co.uk',    '230249147'),
    ('NI030990', 'ENCIRC',             'encirc360.com',            '143776493'),
    ('04084819', 'ROPE & SLING',       'ropeandsling.co.uk',       '811037280'),
    ('09066663', 'ABQ INVESTMENT',     'abqglobal.com',            '208066915'),
    ('11668305', 'JERA NEX',           'jeranex.com',              '315775783'),
    ('09098453', 'ELIZABETH SCHOOL',   'esl.ac.uk',                '264050919'),
]
if FIRME_TEST:
    FIRME = [f for f in FIRME if f[1] in FIRME_TEST]

# ------------------------------------------------------------------
# 1. SUMA DE CONTROL, TIPARE, PRIORITATE
# ------------------------------------------------------------------
def tva_valid_checksum(nr):
    nr = nr.replace('GB', '').replace(' ', '')
    if len(nr) != 9 or not nr.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr[:7], range(8, 1, -1))) + int(nr[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

TIPAR = re.compile(r'VAT[^0-9]{0,40}?(?:GB)?\s?(\d{3})\s?(\d{4})\s?(\d{2})(?!\d)', re.I)

def scor_prioritate(url):
    u = url.lower()
    if re.search(r'terms|condition|contact|legal|imprint|about-us|company-info', u):
        return 0
    if re.search(r'privacy|cookie|disclaimer', u):
        return 1
    if re.search(r'polic', u):
        return 2
    return 3

def text_curat(text):
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', text))

def numar_firma_in_context(company_number, context):
    fara_zero = company_number.lstrip('0')
    return bool(re.search(rf'(?<!\d)0*{re.escape(fara_zero)}(?!\d)', context))

# ------------------------------------------------------------------
# 2. CERERI HTTP CU REINCERCARI
# ------------------------------------------------------------------
def get(url, params=None, extra_headers=None):
    headers = {**HEADERS, **(extra_headers or {})}
    for incercare in range(3):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=120)
            if r.status_code in (200, 206, 404):
                return r
            print(f'      status {r.status_code}, reincerc...')
        except requests.RequestException as e:
            print(f'      eroare retea ({type(e).__name__}), reincerc...')
        time.sleep(5 * (incercare + 1))
    return None

# ------------------------------------------------------------------
# 3. INDEXUL: pagini HTML si PDF de pe domeniu (inclusiv subdomenii)
# ------------------------------------------------------------------
def pagini_din_index(crawl, domeniu):
    r = get(f'https://index.commoncrawl.org/{crawl}-index',
            params={'url': domeniu, 'matchType': 'domain', 'output': 'json', 'limit': 1000})
    if r is None:
        return None
    if r.status_code == 404:
        return []
    pagini, vazute = [], set()
    for linie in r.text.splitlines():
        if not linie.strip():
            continue
        p = json.loads(linie)
        mime = p.get('mime', '')
        if p.get('status') == '200' and ('html' in mime or 'pdf' in mime) and p['url'] not in vazute:
            vazute.add(p['url'])
            pagini.append(p)
    pagini.sort(key=lambda p: (scor_prioritate(p['url']), 'pdf' not in p.get('mime', ''), len(p['url'])))
    return pagini

# ------------------------------------------------------------------
# 4. DESCARCAREA UNEI PAGINI (HTML sau PDF) DIN ARHIVA
# ------------------------------------------------------------------
def descarca_pagina(p):
    start = int(p['offset']); end = start + int(p['length']) - 1
    r = get(f"https://data.commoncrawl.org/{p['filename']}",
            extra_headers={'Range': f'bytes={start}-{end}'})
    if r is None:
        return None
    for rec in ArchiveIterator(io.BytesIO(r.content)):
        if rec.rec_type != 'response':
            continue
        continut = rec.content_stream().read()
        if 'pdf' in p.get('mime', ''):
            try:
                pdf = PdfReader(io.BytesIO(continut))
                return ' '.join((pag.extract_text() or '') for pag in pdf.pages)
            except Exception:
                return 'PDF_ILIZIBIL'
        return continut.decode('utf-8', errors='ignore')
    return ''

# ------------------------------------------------------------------
# 5. BUCLA PRINCIPALA
# ------------------------------------------------------------------
print('Crawl:', CRAWL, '\n')
rezultate = []

for nr_firma, firma, domeniu, tva_hmrc in FIRME:
    t0 = time.time()
    print(f'--- {firma} ({domeniu})')
    pagini = pagini_din_index(CRAWL, domeniu)

    rand = {'company_number': nr_firma, 'firma': firma, 'domeniu': domeniu, 'tva_hmrc': tva_hmrc,
            'pagini_in_index': '', 'pdf_in_index': 0, 'pagini_descarcate': 0,
            'pagini_cu_cuvantul_vat': 0, 'pdf_ilizibile': 0, 'erori_descarcare': 0,
            'tva_gasit_cc': False, 'url_gasit': '', 'tip_pagina': '', 'pozitie_in_lista': '',
            'nr_firma_langa_tva': False, 'alte_numere': '', 'diagnostic': '', 'secunde': 0}

    if pagini is None:
        rand['pagini_in_index'] = 'EROARE'
        rand['diagnostic'] = 'eroare server la index - de reincercat'
    elif not pagini:
        rand['pagini_in_index'] = 0
        rand['diagnostic'] = 'domeniul lipseste din crawl'
    else:
        rand['pagini_in_index'] = len(pagini)
        rand['pdf_in_index'] = sum('pdf' in p.get('mime', '') for p in pagini)
        alte = {}
        for i, p in enumerate(pagini[:MAX_PAGINI]):
            text = descarca_pagina(p)
            if text is None:
                rand['erori_descarcare'] += 1
                continue
            if text == 'PDF_ILIZIBIL':
                rand['pdf_ilizibile'] += 1
                print(f'    PDF ilizibil (posibil trunchiat): {p["url"]}')
                continue
            rand['pagini_descarcate'] += 1
            if re.search(r'\bVAT\b', text, re.I):
                rand['pagini_cu_cuvantul_vat'] += 1
            for m in TIPAR.finditer(text):
                nr = ''.join(m.groups())
                if not tva_valid_checksum(nr):
                    continue
                context = text_curat(text[max(0, m.start()-250): m.end()+80])[-220:]
                if nr == tva_hmrc:
                    rand['tva_gasit_cc'], rand['url_gasit'] = True, p['url']
                    rand['tip_pagina'] = 'pdf' if 'pdf' in p.get('mime', '') else 'html'
                    rand['pozitie_in_lista'] = i
                    rand['nr_firma_langa_tva'] = numar_firma_in_context(nr_firma, context)
                    print(f'    CORECT ({rand["tip_pagina"]}, pagina nr. {i}) pe {p["url"]}\n      ...{context}...')
                elif nr not in alte:
                    alte[nr] = p['url']
                    print(f'    ALT NUMAR {nr} pe {p["url"]}\n      ...{context}...')
            if rand['tva_gasit_cc']:
                break
            time.sleep(1)
        rand['alte_numere'] = ' '.join(f'{n}@{u}' for n, u in alte.items())

        if rand['tva_gasit_cc']:
            rand['diagnostic'] = 'gasit'
        elif rand['pagini_cu_cuvantul_vat'] == 0:
            rand['diagnostic'] = 'nicio pagina descarcata nu contine cuvantul VAT'
        else:
            rand['diagnostic'] = 'apare cuvantul VAT, dar nu numarul firmei - de verificat de mana'

    rand['secunde'] = round(time.time() - t0)
    rezultate.append(rand)
    print(f"    index={rand['pagini_in_index']} (pdf={rand['pdf_in_index']}) "
          f"descarcate={rand['pagini_descarcate']} cu_VAT={rand['pagini_cu_cuvantul_vat']} "
          f"pdf_ilizibile={rand['pdf_ilizibile']} erori={rand['erori_descarcare']} "
          f"gasit={rand['tva_gasit_cc']} nr_firma_langa={rand['nr_firma_langa_tva']} "
          f"{rand['secunde']}s -> {rand['diagnostic']}\n")
    time.sleep(2)

# ------------------------------------------------------------------
# 6. REZUMAT + SALVARE
# ------------------------------------------------------------------
n = len(rezultate)
erori = sum(r['pagini_in_index'] == 'EROARE' for r in rezultate)
prezente = sum(isinstance(r['pagini_in_index'], int) and r['pagini_in_index'] > 0 for r in rezultate)
gasite = sum(r['tva_gasit_cc'] for r in rezultate)
print(f'Domenii prezente in {CRAWL}: {prezente}/{n}  (erori de server: {erori})')
print(f'TVA corect gasit in Common Crawl: {gasite}/{n}')
print(f'Timp total: {sum(r["secunde"] for r in rezultate) // 60} min')

with open(FISIER_IESIRE, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rezultate[0].keys()))
    w.writeheader(); w.writerows(rezultate)
print('Salvat:', FISIER_IESIRE)

Crawl: CC-MAIN-2026-39 

--- GREENSPRUCE GP (invesco.com)
    index=865 (pdf=0) descarcate=30 cu_VAT=0 pdf_ilizibile=0 erori=0 gasit=False nr_firma_langa=False 73s -> nicio pagina descarcata nu contine cuvantul VAT

--- BP3 GLOBAL (bp-3.com)
      status 504, reincerc...
      status 504, reincerc...
    index=229 (pdf=1) descarcate=30 cu_VAT=0 pdf_ilizibile=0 erori=0 gasit=False nr_firma_langa=False 105s -> nicio pagina descarcata nu contine cuvantul VAT

Domenii prezente in CC-MAIN-2026-39: 2/2  (erori de server: 0)
TVA corect gasit in Common Crawl: 0/2
Timp total: 2 min
Salvat: ../data/cc_pas1_v3_retry.csv
